> **Sentetik veri.** Bu defter `python -m sentetik.uret` ile üretilen uydurma veriyle çalışır. Sayılar gerçek bir şirkete ait değildir ve gerçek projenin sonuçlarını temsil etmez; amaç yöntemi uçtan uca göstermektir. *(Synthetic data: numbers are not real results.)*

# Sinyal — 06 · İleri test (2026-04..07) ve canlı tahmin (2026-10-01)

Tanımlar: `docs/preregistration.md` **v1.3** (sonuçlar görülmeden yazıldı). Ön koşul: `egitim_a_ileri` / `egitim_b_ileri` tabloları (sentetik veride hazır).

1. **İleri test (tek sefer):** 03'te dondurulmuş test2 modelleri, hiç görmedikleri ve etiketi kapanmış 2026-04..07 kesimlerinde. Kilit: `docs/sonuclar/06_ILERI_ACILDI.txt`.
2. **Canlı tahmin:** 2026-10-01 kesimi (Ekim–Aralık 2026 alıcıları). Yeniden eğitilmiş model skorlar; skorlar BigQuery'de **üzerine yazılamaz** tabloya yazılır ve SHA-256 ile mühürlenir.
3. **Ara okuma:** Ekim 2026 bittikten sonra `ilk_alim` tablosu üretilir (`sql/24_ara_okuma.sql`; sentetik veride hazır) ve yalnızca bölüm 3 (ya da `scripts/07_ara_okuma.py`) çalıştırılır.

**Çalıştırma:** Run All. İleri test ilk çalıştırmada açılır, sonrakilerde kayıtlı sonuçlar okunur. Bölüm 2, GA4 verisi canlı kesimden bir gün öncesine kadar gelmiyorsa **durur** (koruma). Bölüm 3, `ilk_alim` tablosu yoksa kendini atlar.

In [ ]:
import sys, json, hashlib, datetime as dt
from pathlib import Path
KOK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(KOK))

import joblib
import numpy as np
import pandas as pd

from src.veri import tablo_yukle, KAYNAK, PROJE, VERI_SETI
from src.ozellikler import hazirla, model_a_ozellikleri, model_a_bayrak_ozellikleri, model_b_ozellikleri
from src.degerlendirme import (metrikler, alt_kumeler, bootstrap_fark, capture_at_k, isaret_tutarli,
                               hedef_egitim_indeksleri, ISARET_MIN_KAT)
from src.modeller import baz_kural, baz_rfm, baz_bgnbd, egit, skorla
from src.kalibrasyon import duzey_kaydir

SONUC = KOK / "docs" / "sonuclar"; VERI = KOK / "data"; MODEL_DIZIN = VERI / "modeller_03"
TOHUM, BOOTSTRAP_N = 42, 1000
ILERI = [pd.Timestamp(x) for x in ["2026-04-01", "2026-05-01", "2026-06-01", "2026-07-01"]]
CANLI = pd.Timestamp("2026-10-01")
KILIT = SONUC / "06_ILERI_ACILDI.txt"
pd.set_option("display.float_format", "{:.4f}".format)

## 0. Veri: eski + yeni kesimler (aynı hazırlık)

In [ ]:
a0, b0 = tablo_yukle("egitim_a"), tablo_yukle("egitim_b")
a1, b1 = tablo_yukle("egitim_a_ileri"), tablo_yukle("egitim_b_ileri")
assert set(a0.columns) == set(a1.columns) and set(b0.columns) == set(b1.columns), "kolonlar farklı"
assert not set(a0["kesim"]) & set(a1["kesim"]), "kesimler çakışıyor"
d0, d1 = hazirla(a0, b0), hazirla(a1, b1)          # endeks ve göreli davranış kesim içinde → ayrı hazırlamak güvenli
df = pd.concat([d0, d1[d0.columns]], ignore_index=True)
df["y_etiket"] = df["y_etiket"].astype("float")      # canlı kesimde NaN; diğerlerinde 0/1
assert df.loc[df["kume"] == "canli", "y_etiket"].isna().all(), "canlı kesimde etiket olmamalı"
assert not df.loc[df["kume"] != "canli", "y_etiket"].isna().any()

SETLER = {"A": model_a_ozellikleri(df), "A+bayrak": model_a_bayrak_ozellikleri(df), "B": model_b_ozellikleri(df)}
assert [len(v) for v in SETLER.values()] == [24, 25, 38]
akt = df["davranis_var_30g"].fillna(False).astype(bool)
son_gun = df[akt].groupby("kesim")["gun_son_aktivite"].min()          # 1 = davranış verisi kesimden bir gün önceye kadar var
print("davranış verisinin bittiği gün (kesimden önce):", son_gun.astype(int).to_dict())
assert (son_gun.drop(CANLI, errors="ignore") == 1).all(), "canlı dışı bir kesimde GA4 verisi eksik"
(df.groupby(["kesim", "kume"]).agg(etiket_orani=("y_etiket", "mean"), davranis_kapsami=("davranis_var_30g", "mean"))
   .reset_index().assign(kesim=lambda t: t["kesim"].dt.strftime("%Y-%m")))

## 1. İleri test — dondurulmuş test2 modelleri (TEK SEFER)

In [ ]:
MODEL = {s: joblib.load(MODEL_DIZIN / f"test2_lightgbm_{s.replace('+', '_')}.joblib") for s in SETLER}
ILERI_YAPILDI = KILIT.exists()
if ILERI_YAPILDI:      # tek sefer kuralı: yeniden hesaplanmaz, kayıtlı sonuçlar okunur
    print("İleri test daha önce açıldı → kayıtlı sonuçlar yükleniyor.\n" + KILIT.read_text(encoding="utf-8"))
    MI = pd.read_csv(SONUC / "06_ileri_metrikler.csv"); FI = pd.read_csv(SONUC / "06_ileri_bootstrap.csv")
else:
    KILIT.write_text(f"İleri test ilk kez açıldı: {dt.datetime.now().isoformat(timespec='seconds')}\n"
                     f"Model: dondurulmuş test2 LightGBM (2024-10 → 2025-10)\n", encoding="utf-8")

    satir, fark, SKOR_I = [], [], {}
    for k in ILERI:
        d = df[df["kesim"] == k]; y = d["y_etiket"].astype(int).to_numpy()
        sk = {f"lightgbm|{s}": skorla("lightgbm", MODEL[s], d, kol) for s, kol in SETLER.items()}
        sk["kural|-"], sk["rfm|-"] = baz_kural(d).to_numpy(), baz_rfm(d).to_numpy()
        try:
            sk["bgnbd|-"] = baz_bgnbd(d).to_numpy()
        except Exception as hata:
            print("BG/NBD atlandı:", hata)
        SKOR_I[k] = sk
        for ad, s in sk.items():
            for ak, m in alt_kumeler(d).items():
                satir.append({"kesim": k.date(), "model": ad.split("|")[0], "set": ad.split("|")[1], "alt_kume": ak,
                              **metrikler(y[m], s[m], olasilik=ad.startswith("lightgbm"))})
        r = bootstrap_fark(y, sk["lightgbm|A"], sk["lightgbm|B"], metrik=capture_at_k, n=BOOTSTRAP_N, tohum=TOHUM)
        fark.append({"kesim": k.date(), **r}); print(k.date(), "tamam")
    MI = pd.DataFrame(satir); MI.to_csv(SONUC / "06_ileri_metrikler.csv", index=False)
    FI = pd.DataFrame(fark);  FI.to_csv(SONUC / "06_ileri_bootstrap.csv", index=False)
(FI.set_index("kesim") * 100).round(2)

In [ ]:
t = MI.query("alt_kume == 'tum'")
tablo = t.pivot_table(index=["model", "set"], columns="kesim", values="capture@10")
display(tablo)
en_iyi_baz = t[t["model"].isin(["kural", "rfm", "bgnbd"])].groupby("kesim")["capture@10"].max()
b_cap = t.query("model == 'lightgbm' and set == 'B'").set_index("kesim")["capture@10"]
ref = pd.read_csv(SONUC / "03_karar.csv")
test_farki = float(pd.read_csv(SONUC / "03_bootstrap.csv")
                   .query("bolum == 'test' and model == 'lightgbm' and alt_kume == 'tum' and metrik == 'capture@10'")["fark"].iloc[0])
K1 = isaret_tutarli(FI["fark"].to_numpy(), test_farki=test_farki, min_kat=ISARET_MIN_KAT)
K2 = bool((b_cap > en_iyi_baz).all())
KI = pd.DataFrame([
    ("B − A, 4 kesimin ≥ 3'ünde testle aynı işaret", " ".join(f"{x*100:+.2f}" for x in FI["fark"]), K1),
    ("B her kesimde en iyi bazı geçer (capture@10)", " ".join(f"{x:.3f}>{y_:.3f}" for x, y_ in zip(b_cap, en_iyi_baz)), K2),
], columns=["olcut", "deger", "saglandi"])
KI.to_csv(SONUC / "06_ileri_karar.csv", index=False)
print("İLERİ TEST:", "DESTEKLER" if (K1 and K2) else "DESTEKLEMEZ")
KI

## 2. Canlı tahmin — 2026-10-01 (Ekim–Aralık 2026)

In [ ]:
KUMELER = ("egitim", "tampon", "test", "test2", "ek_egitim", "ileri")
egt = hedef_egitim_indeksleri(df, CANLI, KUMELER)
assert not df.loc[egt, "y_etiket"].isna().any()
print("canlı model eğitim kesimleri:", sorted(df.loc[egt, "kesim"].dt.strftime("%Y-%m").unique()))
c = df[df["kesim"] == CANLI].copy()
son = int(c.loc[c["davranis_var_30g"].fillna(False).astype(bool), "gun_son_aktivite"].min())
assert son == 1, (f"Davranış verisi canlı kesimden {son} gün önce bitiyor (7 günlük özellikler boş, 30 günlükler eksik). "
                  "Davranış kaynağını kesimden bir gün öncesine kadar tamamlayın ve önbelleği yenileyin: "
                  "tablo_yukle('egitim_b_ileri', yenile=True).")
CM = {}
for s in ["A", "B"]:
    CM[s] = egit("lightgbm", df.loc[egt], SETLER[s])
    joblib.dump(CM[s], VERI / "modeller_03" / f"canli_20261001_lightgbm_{s}.joblib")
c["skor_B"] = skorla("lightgbm", CM["B"], c, SETLER["B"])
c["skor_A"] = skorla("lightgbm", CM["A"], c, SETLER["A"])
c["skor_B_dondurulmus"] = skorla("lightgbm", MODEL["B"], c, SETLER["B"])
hedef = float(df.loc[df["kesim"] == pd.Timestamp("2025-10-01"), "y_etiket"].mean())
c["p_kalibre"] = duzey_kaydir(c["skor_B"].to_numpy(), hedef)              # v1.3: yalnızca M2
c["sira_B"] = c["skor_B"].rank(ascending=False, method="first").astype(int)
c["ust_yuzde10"] = c["sira_B"] <= int(round(0.10 * len(c)))
print("hedef düzey (2025-10 oranı):", round(hedef, 4), "| skor korelasyonu (yeni vs dondurulmuş B):",
      round(np.corrcoef(c["skor_B"], c["skor_B_dondurulmus"])[0, 1], 3))

In [ ]:
# Mühür + BigQuery (üzerine yazılamaz)
zaman = dt.datetime.now().isoformat(timespec="seconds")
cikti = c[["kesim", "musteri_id", "skor_B", "skor_A", "skor_B_dondurulmus", "p_kalibre", "sira_B", "ust_yuzde10"]].copy()
cikti["skorlama_zamani"] = zaman
cikti["model"] = "LightGBM B, v1.3, eğitim 2024-10→2026-07"
metin = "\n".join(f"{r.musteri_id}|{r.skor_B:.8f}" for r in cikti.sort_values("musteri_id").itertuples())
muhur = hashlib.sha256(metin.encode("utf-8")).hexdigest()
cikti.to_parquet(VERI / "06_canli_skor_20261001.parquet", index=False)

if KAYNAK == "bigquery":   # gerçek veride: üzerine yazılamaz tablo (WRITE_EMPTY)
    from google.cloud import bigquery
    istemci = bigquery.Client(project=PROJE)
    hedef_tablo = f"{PROJE}.{VERI_SETI}.canli_skor_20261001"
    is_ = istemci.load_table_from_dataframe(
        cikti, hedef_tablo, job_config=bigquery.LoadJobConfig(write_disposition="WRITE_EMPTY"))
    is_.result()
else:                       # sentetik veride: yalnızca yerel Parquet + mühür
    hedef_tablo = str(VERI / "06_canli_skor_20261001.parquet")
bilgi = {"zaman": zaman, "sha256": muhur, "tablo": hedef_tablo, "kesim": "2026-10-01",
         "hedef_pencere": "2026-10-01 → 2026-12-30", "kalibrasyon": "M2, hedef = 2025-10 etiket oranı",
         "ozellikler_B": SETLER["B"]}
(SONUC / "06_canli_muhur.json").write_text(json.dumps(bilgi, ensure_ascii=False, indent=1), encoding="utf-8")
print("MÜHÜR (kayda geçir):", muhur)

In [ ]:
# Sunum için maskeli özet: olasılık bantları (popülasyon payı) — sayı yok
bant = pd.cut(c["p_kalibre"], [-1, 0.10, 0.30, 2], labels=["düşük (<%10)", "orta (%10–30)", "yüksek (≥%30)"])
OZET = (pd.DataFrame({"bant": bant, "p": c["p_kalibre"], "davranis": c["davranis_var_30g"].astype(bool)})
          .groupby("bant", observed=True).agg(pay=("p", "size"), beklenen_alim=("p", "mean"), davranis_payi=("davranis", "mean")))
OZET["pay"] = OZET["pay"] / len(c)
OZET.to_csv(SONUC / "06_canli_ozet.csv")
OZET

## 3. Ara okuma (1–2 Kasım'dan sonra; önce `sql/24_ara_okuma.sql`)

In [ ]:
try:
    ilk = tablo_yukle("ilk_alim", yenile=True)
except Exception as hata:
    ilk = None; print("ilk_alim tablosu yok — ara okuma atlandı:", str(hata)[:120])

if ilk is not None:
    ilk["ilk_alim_tarihi"] = pd.to_datetime(ilk["ilk_alim_tarihi"])
    st = pd.read_parquet(VERI / "03_skorlar_test.parquet")
    cs = pd.read_parquet(VERI / "06_canli_skor_20261001.parquet")
    ao, YB = [], {}
    for ad, skorlar, kesim, kolA, kolB in [("kıyas: Ekim 2025 (test)", st, pd.Timestamp("2025-10-01"), "skor_lightgbm_A", "skor_lightgbm_B"),
                                          ("canlı: Ekim 2026", cs, CANLI, "skor_A", "skor_B")]:
        x = skorlar.merge(ilk[ilk["kesim"] == kesim][["musteri_id", "ilk_alim_tarihi"]], on="musteri_id", how="left", validate="one_to_one")
        y_ay = (x["ilk_alim_tarihi"] < kesim + pd.DateOffset(months=1)).astype(int).to_numpy()   # ilk ayda alanlar
        YB[ad] = (y_ay, x[kolA].to_numpy(), x[kolB].to_numpy())
        ao.append({"donem": ad, "ilk_ay_alim_orani": y_ay.mean(),
                   "capture@10_B": capture_at_k(y_ay, x[kolB]), "capture@10_A": capture_at_k(y_ay, x[kolA]),
                   "son_veri": x["ilk_alim_tarihi"].max()})
    AO = pd.DataFrame(ao); AO["B-A_puan"] = (AO["capture@10_B"] - AO["capture@10_A"]) * 100
    AO.to_csv(SONUC / "06_ara_okuma.csv", index=False)
    display(AO)

    # tanimlar v1.5 — ara okuma ölçütleri (ara okumadan önce yazıldı)
    kiyas, canli = AO.iloc[0], AO.iloc[1]
    y_c, a_c, b_c = YB["canlı: Ekim 2026"]
    bs = bootstrap_fark(y_c, a_c, b_c, metrik=capture_at_k, n=BOOTSTRAP_N, tohum=TOHUM)
    O1 = canli["capture@10_B"] >= kiyas["capture@10_B"] - 0.03
    O2 = canli["capture@10_B"] - canli["capture@10_A"] > 0
    KA = pd.DataFrame([
        ("Ölçüt 1: B_canlı ≥ B_kıyas − 3 puan", f"{canli['capture@10_B']:.3f} vs {kiyas['capture@10_B']:.3f}", bool(O1)),
        ("Ölçüt 2: B − A > 0 (canlı)", f"{(canli['capture@10_B'] - canli['capture@10_A']) * 100:+.2f} puan "
                                       f"[{bs['alt_%2.5'] * 100:+.2f}, {bs['ust_%97.5'] * 100:+.2f}]", bool(O2)),
    ], columns=["olcut", "deger", "saglandi"])
    KA.to_csv(SONUC / "06_ara_okuma_karar.csv", index=False)
    print("ARA OKUMA:", "DESTEKLER" if (O1 and O2) else "DESTEKLEMEZ", "(bilgi amaçlı; kesin okuma 30 Aralık sonrası)")
    display(KA)